In [1]:
pip install langchain-groq langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.9 MB/s eta 0:00:00


In [2]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
from google.colab import userdata
from langgraph.graph import StateGraph , START , END
from typing import TypedDict
from pydantic import BaseModel
from langgraph.checkpoint.memory import InMemorySaver

api_key = userdata.get("groq_api_key")

model = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=api_key
)


In [3]:
class str_schema(BaseModel):
    question:str
    city:str
    population:int

city_population_extractor = model.with_structured_output(str_schema)

class simple_state(TypedDict):
    question:str
    city:str
    population:str

In [4]:
def city(state:simple_state):
    question = state["question"]
    prompt = f"Based on the question: '{question}', please provide the name of the city."
    response = city_population_extractor.invoke(prompt)
    return {"city":response.city}

def population(state:simple_state):
    city_name = state["city"]
    prompt = f"Based on the city: '{city_name}', please provide the population of that city."
    response = city_population_extractor.invoke(prompt)
    return {"population":response.population}

In [5]:
graph = StateGraph(simple_state)


graph.add_node("city" , city)
graph.add_node("population" , population)


graph.add_edge(START , "city")
graph.add_edge("city" , "population")
graph.add_edge("population" , END)

In [7]:
checkpoint = InMemorySaver()
workflow = graph.compile(checkpointer=checkpoint)

config = {
    "configurable":{
        "thread_id": "1"
    }
}

In [11]:
res = workflow.invoke( {"question":"What is the capital of Pakistan?"},config=config)
print(res)

{'question': 'What is the capital of Pakistan?', 'city': 'Islamabad', 'population': 1065200}


In [9]:
config2 = {
    "configurable":{
        "thread_id": "2"
    }
}

In [12]:
res2 = workflow.invoke( {"question":"What is the capital of India?"},config=config2)
print(res2)

{'question': 'What is the capital of India?', 'city': 'New Delhi', 'population': 24999999}


In [13]:
output = workflow.get_state(config2)
print(output)

StateSnapshot(values={'question': 'What is the capital of India?', 'city': 'New Delhi', 'population': 24999999}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f192253-200b-6dc1-8002-d8d84e4549a4'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-07T05:59:47.555029+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f192253-18c8-66ca-8001-dda98e1387c9'}}, tasks=(), interrupts=())


In [14]:
list(workflow.get_state_history(config))

[StateSnapshot(values={'question': 'What is the capital of Pakistan?', 'city': 'Islamabad', 'population': 1065200}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f192251-e38a-635f-8008-96cf83f3b364'}}, metadata={'source': 'loop', 'step': 8, 'parents': {}}, created_at='2026-08-07T05:59:14.366933+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f192251-e162-67cc-8007-15272add0c64'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'question': 'What is the capital of Pakistan?', 'city': 'Islamabad'}, next=('population',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f192251-e162-67cc-8007-15272add0c64'}}, metadata={'source': 'loop', 'step': 7, 'parents': {}}, created_at='2026-08-07T05:59:14.140936+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f192251-df85-668c-8006-ec8a9b095f6c'}}, tasks=(PregelTask(id='20

# Time Travel

In [15]:
class joke_schema(BaseModel):
  joke:str
  explaintion:str

res = model.with_structured_output(joke_schema)

In [16]:
class joke_state(TypedDict):
  topic:str
  joke:str
  explaintion:str


In [17]:
def joke(state:joke_state):
  topic = state["topic"]
  prompt = f"Tell me a joke about {topic}."
  response = res.invoke(prompt)
  return {"joke":response.joke}

def explaination(state:joke_state):
  joke = state["joke"]
  prompt = f"Explain the joke: '{joke}'."
  response = res.invoke(prompt)
  return {"explaintion":response.explaintion}

In [18]:
graph = StateGraph(joke_state)

graph.add_node("gen_joke" , joke)

graph.add_node("gen_explaintion" , explaination)

graph.add_edge(START , "gen_joke")
graph.add_edge("gen_joke" , "gen_explaintion")
graph.add_edge("gen_explaintion" , END)

In [19]:
checkpoint = InMemorySaver()
workflow = graph.compile(checkpointer=checkpoint)

In [20]:
output = workflow.invoke({"topic":"Scool"} , {"configurable":{"thread_id":1}})
print(output)

{'topic': 'Scool', 'joke': 'Why did the student bring a ladder to school?', 'explaintion': 'There is no punchline so the reason for the ladder might be that it is being used for some construction work in the school or that the student is very tall and needs it to reach the top shelf in the classroom.'}


In [21]:
workflow.get_state({"configurable":{"thread_id":"1"}})

StateSnapshot(values={'topic': 'Scool', 'joke': 'Why did the student bring a ladder to school?', 'explaintion': 'There is no punchline so the reason for the ladder might be that it is being used for some construction work in the school or that the student is very tall and needs it to reach the top shelf in the classroom.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f192257-e9fd-6338-8002-70c2817ecc4c'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-07T06:01:56.104451+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f192257-e605-6a77-8001-a6cb9c6c246f'}}, tasks=(), interrupts=())

In [22]:
list(workflow.get_state_history({"configurable":{"thread_id":"1"}}))

[StateSnapshot(values={'topic': 'Scool', 'joke': 'Why did the student bring a ladder to school?', 'explaintion': "This is a play on words. The answer is likely that the student needed to reach a high level of learning. It is a pun on the phrase 'high level of learning' and the idea of literally needing a ladder to reach something. The joke is that the word 'high' has a double meaning, both referring to the difficulty of the material and the need for a physical ladder to access something."}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f191b96-c625-6a23-8002-441a6b1a5a6a'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-08-06T17:08:19.855795+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f191b96-b9a2-60b6-8001-138b5c53c282'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'Scool', 'joke': 'Why did the student bring a ladder to school?'}, next=('gen_expl

In [22]:
workflow.get_state({"configurable":{"thread_id":1 , "checkpoint_id":"1f1918b8-98b5-6cd4-801b-7f44d76d0dfb"}})

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1918b8-98b5-6cd4-801b-7f44d76d0dfb'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [28]:
output = workflow.invok(None ,{"configurable":{"thread_id":1, "checkpoint_id":"1f1918a3-9ef5-6c79-800b-7034e0007cbc"}})
print(output["topic"])
print(output["joke"])
print(output["explaintion"])

Scool
Why did the math book look so sad? Because it had too many problems. Why did the kid bring a ladder to school? He wanted to reach his full potential.
This joke is a play on words. The phrase 'too many problems' has a double meaning. In math, problems refer to mathematical equations. However, the phrase 'too many problems' is also an idiom that means having too many issues or difficulties. The punchline connects the math book's problems to the idea of the kid's 'full potential', implying that the kid wants to achieve his goals and reach new heights, much like the ladder helps him do. The joke relies on wordplay, using the multiple meanings of 'problems' to create a clever connection between the two parts.


In [29]:
workflow.update_state({"configurable":{"thread_id":1 , "checkpoint_id":"1f1918a3-9ef5-6c79-800b-7034e0007cbc" ,"checkpoint_ns":"" }} , {"topic":"Laptop"})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f192261-2d6a-6050-8000-7def55f0e4ad'}}

In [30]:
workflow.get_state({"configurable":{"thread_id":1}})

StateSnapshot(values={'topic': 'Laptop'}, next=('gen_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f192261-2d6a-6050-8000-7def55f0e4ad'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-08-07T06:06:04.766397+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1918a3-9ef5-6c79-800b-7034e0007cbc'}}, tasks=(PregelTask(id='640d012b-a412-8b82-4acb-dc3bd63f6311', name='gen_joke', path=('__pregel_pull', 'gen_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=())

In [37]:
res = workflow.invoke(None ,{"configurable":{"thread_id":1}})
print(["topic"])
print(["joke"])
print(["explaintion"])

Laptop
A laptop is like you, it gets slower and slower over time.
The joke is making a humorous comparison between a laptop and a person, suggesting that both get slower as time passes. The punchline is a play on words, implying that a laptop, like a person, experiences a decline in performance over time.
